In [1]:
import pandas as pd
import numpy as np
import random
import seaborn as sns
import matplotlib.pyplot as plt

import math
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve, roc_curve, auc
from sklearn.model_selection import train_test_split, LeaveOneOut, StratifiedKFold, LeaveOneOut




# Carregar el dataset
df = pd.read_csv('FinalStemmedSentimentAnalysisDataset.csv',sep=';')
df


,tweetId,tweetText,tweetDate,sentimentLabel
0,1,is so sad for my apl friend,04/03/2015,0
1,2,i miss the new moon trail,06/10/2015,0
2,3,omg it already 730 o,03/04/2015,1
3,4,omgag im sooo im gunn cry i've been at thi de...,13/11/2015,0
4,5,i think mi bf is che on me tt,10/08/2015,0
...,...,...,...,...
1564297,1578623,zzzzzz fin night tweet,03/10/2015,1
1564298,1578624,zzzzzzz sleep wel peopl,20/01/2015,1
1564299,1578625,zzzzzzzzzz wait no i hav homework,16/08/2015,0
1564300,1578626,zzzzzzzzzzzzz meh what am i doing up again,03/02/2015,0


In [2]:
# Preprocessament: dividir els texts dels tweets en llistes de paraules

df['tweetText'] = df['tweetText'].str.lower().str.split() #Tot minúscules i concatenació
df_sense_nans = df.dropna(subset=['tweetText'])

In [3]:
#Partició inicial (20% test)

porcio_test = 0.2

X_train, X_test, y_train, y_test = train_test_split(df_sense_nans['tweetText'], df_sense_nans['sentimentLabel'], test_size=porcio_test, stratify=df_sense_nans['sentimentLabel'], random_state=42)


In [4]:
def fer_diccionari(X_train, y_train):
    diccionari = {}
    
    for text, classe in zip(X_train, y_train):
        if isinstance(text, list):  
            for paraula in text:
                if paraula not in diccionari:
                    diccionari[paraula] = {'pos': 0, 'neg': 0}  # Diccionari d'un diccionari

                if classe == 1:  # Positiu
                    diccionari[paraula]['pos'] += 1
                elif classe == 0:  # Negatiu
                    diccionari[paraula]['neg'] += 1
        else:
            print(f"Valor invàlid trobat en el text: {text}")  

    
    return diccionari


diccionari = fer_diccionari(X_train, y_train)



In [5]:
#Calcular les probabilitats de les classes (positiu i negatiu)


def calculProbs(y_train, diccionari, laplace = False):

    if laplace is False:
        total_pos = sum(y_train == 1)
        total_neg = sum(y_train == 0)

        #Càlcul de les probabilitats de les paraules per cada classe
        prob_classes = {'pos': total_pos / len(y_train), 'neg': total_neg / len(y_train)} #Probabilitats positiva i negativa total

        prob_paraules = {}
        for paraula, valor in diccionari.items():
            prob_pos = (valor['pos']) / (total_pos)
            prob_neg = (valor['neg']) / (total_neg)
            prob_paraules[paraula] = {'pos': prob_pos, 'neg': prob_neg} #Diccionari de probabilitats per a cada paraula
        
        return prob_paraules, prob_classes

    else: #Laplace Smoothing

        total_pos = sum(y_train == 1)
        total_neg = sum(y_train == 0)

    
        prob_classes = {'pos': total_pos / len(y_train), 'neg': total_neg / len(y_train)}

        prob_paraules = {}
        
        
        for paraula, valor in diccionari.items():
            
            prob_pos = (valor['pos'] + 1) / (total_pos + len(diccionari))  
            prob_neg = (valor['neg'] + 1) / (total_neg + len(diccionari))  
            
            prob_paraules[paraula] = {'pos': prob_pos, 'neg': prob_neg}
    
    return prob_paraules, prob_classes



prob_paraules, prob_classes = calculProbs(y_train, diccionari)
#-----------------------------------------------------------------------------------------------------------------------------------------
#Funció per fer prediccions 

def fer_prediccio(twit, prob_classes, prob_paraules):
    if prob_classes['pos'] > 0:
        prob_pos = math.log(prob_classes['pos'])
    else:
        prob_pos = 0

    if prob_classes['neg'] > 0:
        prob_neg = math.log(prob_classes['neg'])
    else:
        prob_neg = 0

    for paraula in twit:
        if paraula in prob_paraules:
            if prob_paraules[paraula]['pos']>0:
                prob_pos += math.log(prob_paraules[paraula]['pos'])
            if prob_paraules[paraula]['neg']>0:
                prob_neg += math.log(prob_paraules[paraula]['neg'])

    if prob_pos > prob_neg:
        return 1  
    else:
        return 0  
    


#Prediccions


prediccions = [] 
for twit in X_test:
    prediccions.append(fer_prediccio(twit, prob_classes, prob_paraules))


#-----------------------------------------------------------------------------------------------------------------------------------------

#Avaluar el model utilitzant mètriques

def metriques(y_pred, y_test):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    f1 = f1_score(y_test, y_pred, average='binary')

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    specificity = tn / (tn + fp)

    return accuracy, precision, recall, f1, specificity

accuracy, precision, recall, f1, specificity = metriques(prediccions, y_test)



# Mostrar els resultats
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)
print("F1-Score: ", f1)
print("Specificity: ", specificity)
print(" ")

#-----------------------------------------------------------------------------------------------------------------------------------------

def corbes(y_test, y_pred):

    #Matriu de confusió

    cm = confusion_matrix(y_test, y_pred)

    # Crear el heatmap
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negativa', 'Positive'])
    plt.xlabel('Prediccions')
    plt.ylabel('Valors reals')
    plt.title('Matriu de Confusió')
    plt.show()



    # Calcula les dades per a la corba Precision-Recall
    precision, recall, _ = precision_recall_curve(y_test, y_pred)
    
    # Calcula les dades per a la corba ROC
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    
    # Calcula l'AUC per a la ROC
    roc_auc = auc(fpr, tpr)

    #Precision-Recall
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(recall, precision, color='blue', linewidth=4)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Corba Precision-Recall')
    plt.ylim([0, 1])
    plt.xlim([0, 1])  # Límits de l'eix X
    plt.grid(True)

    #ROC
    plt.subplot(1, 2, 2)
    plt.plot(fpr, tpr, color='green', linewidth=4, label=f'AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='grey')  # Diagonal de no-discriminació
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR)')
    plt.title('Corba ROC')
    plt.ylim([0, 1])
    plt.xlim([0, 1])
    plt.legend(loc='lower right')
    plt.grid(True)


    plt.tight_layout()
    plt.show()

#corbes(y_test, prediccions)


"""

#Matriu de confusió
cm = confusion_matrix(y_test, prediccions)

# Crear el heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Prediccions')
plt.ylabel('Valors reals')
plt.title('Matriu de Confusió')
plt.show()
"""


Accuracy:  0.6769727925946761
Precision:  0.7535602862910625
Recall:  0.5253283182047886
F1-Score:  0.6190790292001341
Specificity:  0.8284215704438155
 


"\n\n#Matriu de confusió\ncm = confusion_matrix(y_test, prediccions)\n\n# Crear el heatmap\nplt.figure(figsize=(8, 6))\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])\nplt.xlabel('Prediccions')\nplt.ylabel('Valors reals')\nplt.title('Matriu de Confusió')\nplt.show()\n"

In [6]:
def cross_validation(df, k=5, particio_dict = 1, laplace = False):
    kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42) 
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    specificities = []
    

    X = df['tweetText']
    y = df['sentimentLabel']
    
    for train_index, test_index in kf.split(X,y):

        train_data, test_data = df.iloc[train_index], df.iloc[test_index]
        

        X_train, X_test = train_data['tweetText'], test_data['tweetText']
        y_train, y_test = train_data['sentimentLabel'], test_data['sentimentLabel']
        

        diccionari = fer_diccionari(X_train, y_train)

        num_items = len(diccionari)

        if particio_dict > 1:
            num_items = num_items // particio_dict 
            diccionari = dict(random.sample(list(diccionari.items()), num_items))
            
        prob_paraules, prob_classes = calculProbs(y_train, diccionari, laplace) #posar a True si vull Laplace
        

        prediccions = []
        for twit in X_test:
            prediccions.append(fer_prediccio(twit,prob_classes,prob_paraules))

        
        
        accuracy, precision, recall, f1, specificity = metriques(prediccions, y_test)
        
        accuracies.append(accuracy)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        specificities.append(specificity)

        #corbes(y_test, prediccions)


        #print("Accuracy: ", accuracy)
        #print("Precision: ", precision)
        #print("Recall: ", recall)
        #print("F1-Score: ", f1)
        #print(" ")
            
    mitjana_accuracies = np.mean(accuracies)
    mitjana_precisions = np.mean(precisions)
    mitjana_recalls = np.mean(recalls)
    mitjana_f1s = np.mean(f1_scores)
    mitjana_specify = np.mean(specificities)

    #corbes(y_test, prediccions)

    print("AVALUACIÓ AMB K-FOLD CROSS-VALIDATION:")
    print("  Mitjana accuracies:", mitjana_accuracies)
    print("  Mitjana precisions:", mitjana_precisions)
    print("  Mitjana recalls:", mitjana_recalls)
    print("  Mitjana F1-Score:", mitjana_f1s)
    print("  Mitjana Specificities:", mitjana_specify)
    print(" ")

    
    


    return mitjana_accuracies, mitjana_precisions, mitjana_recalls, mitjana_f1s, mitjana_specify, num_items

mitjana_accuracies, mitjana_precisions, mitjana_recalls, mitjana_f1s, mitjana_specify, mida= cross_validation(df_sense_nans, k=5)



AVALUACIÓ AMB K-FOLD CROSS-VALIDATION:
  Mitjana accuracies: 0.6761359858848799
  Mitjana precisions: 0.7521033417038949
  Mitjana recalls: 0.5248453249460405
  Mitjana F1-Score: 0.6182510614493648
  Mitjana Specificities: 0.8272317936682978
 


In [7]:
#PART B
#1.Ampliar particions

particions = [0.2, 0.4, 0.6, 0.8, 0.95]

print("AVALUACIÓ DE L'APARTAT 1 DE LA PART B: AMPLIAR PARTICIONS")
print()

def ampliarParticions(df, particions = [0.2, 0.4, 0.6, 0.8, 0.95], laplace = False):

    #llistes per fer gràfiques
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    specificities = []


    for particio in particions:
        X_train, X_test, y_train, y_test = train_test_split(df['tweetText'], df['sentimentLabel'], train_size=particio, stratify=df['sentimentLabel'], random_state=None)
        
        diccionari = fer_diccionari(X_train, y_train)
        prob_paraules, prob_classes = calculProbs(y_train, diccionari, laplace)
        prediccions = [] 
        for twit in X_test:
            prediccions.append(fer_prediccio(twit, prob_classes, prob_paraules))
        accuracy, precision, recall, f1, specificity = metriques(prediccions, y_test)

        # Afegir les mètriques a les llistes
        accuracies.append(accuracy)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        specificities.append(specificity)

        
        print("  Partició: ",particio, " train, ", 1- particio, " test")
        print("    Accuracy: ", accuracy)
        print("    Precision: ", precision)
        print("    Recall: ", recall)
        print("    F1-Score: ", f1)
        print("    Specificity: ", specificity)
        print(" ")


        #corbes(y_test, prediccions)

        """
    # Crear les gràfiques per cada mètrica
    plt.figure(figsize=(12, 8))
    
    # Gràfic per Accuracy
    plt.subplot(231)
    plt.plot(particions, accuracies, marker='o', linestyle='-', color='b')
    plt.title('Accuracy vs Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('Accuracy')

    # Gràfic per Precision
    plt.subplot(232)
    plt.plot(particions, precisions, marker='o', linestyle='-', color='g')
    plt.title('Precision per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('Precision')

    # Gràfic per Recall
    plt.subplot(233)
    plt.plot(particions, recalls, marker='o', linestyle='-', color='r')
    plt.title('Recall per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('Recall')

    # Gràfic per F1-Score
    plt.subplot(234)
    plt.plot(particions, f1_scores, marker='o', linestyle='-', color='c')
    plt.title('F1-Score per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('F1-Score')

    # Gràfic per Specificity
    plt.subplot(235)
    plt.plot(particions, specificities, marker='o', linestyle='-', color='m')
    plt.title('Specificity per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('Specificity')

    # Mostrar totes les gràfiques
    plt.tight_layout()
    plt.show()"""

ampliarParticions(df_sense_nans)

AVALUACIÓ DE L'APARTAT 1 DE LA PART B: AMPLIAR PARTICIONS

  Partició:  0.2  train,  0.8  test
    Accuracy:  0.6684992456593448
    Precision:  0.7446509735541994
    Recall:  0.512215560626826
    F1-Score:  0.6069408413979462
    Specificity:  0.8245817461648419
 
  Partició:  0.4  train,  0.6  test
    Accuracy:  0.6713759684966886
    Precision:  0.7488251996230719
    Recall:  0.5151093218929511
    F1-Score:  0.6103588979143243
    Specificity:  0.8274416187876129
 
  Partició:  0.6  train,  0.4  test
    Accuracy:  0.6743182166875495
    Precision:  0.7498611512739658
    Recall:  0.5225169036698715
    F1-Score:  0.6158784732628738
    Specificity:  0.8259236307649062
 
  Partició:  0.8  train,  0.19999999999999996  test
    Accuracy:  0.6757421944920347
    Precision:  0.7519025456014248
    Recall:  0.5239465991159556
    F1-Score:  0.6175601296840836
    Specificity:  0.8273418983063842
 
  Partició:  0.95  train,  0.050000000000000044  test
    Accuracy:  0.675160457207149

In [8]:
#2.Diferents mides de diccionari

particions_dict = [1, 2, 3, 4, 5]

print("AVALUACIÓ DE L'APARTAT 2 DE LA PART B: DIFERENTS MIDES DE DICCIONARI")
print(" ")


def diferentsMidesDiccionari(df, particions_dict = [1, 2, 3, 4, 5], laplace = False ):

    #llistes per fer gràfiques
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    specificities = []
    mides = []

    for i in particions_dict:
        
        accuracy, precision, recall, f1, specificity, mida = cross_validation(df, k=5, particio_dict = i, laplace = laplace)

        # Afegir les mètriques a les llistes
        accuracies.append(accuracy)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        specificities.append(specificity)
        mides.append(mida)


        
        print("  Mida del diccionari: ",mida)
        print("    Accuracy: ", accuracy)
        print("    Precision: ", precision)
        print("    Recall: ", recall)
        print("    F1-Score: ", f1)
        print("    Specificity: ", specificity)
        print(" ")

        """
    # Crear les gràfiques per cada mètrica
    plt.figure(figsize=(12, 8))
    
    # Gràfic per Accuracy
    plt.subplot(231)
    plt.plot(mides, accuracies, marker='o', linestyle='-', color='b')
    plt.title('Accuracy per mida de diccionari')
    plt.xlabel('Número de paraules')
    plt.ylabel('Accuracy')

    # Gràfic per Precision
    plt.subplot(232)
    plt.plot(mides, precisions, marker='o', linestyle='-', color='g')
    plt.title('Precision per mida de diccionari')
    plt.xlabel('Número de paraules')
    plt.ylabel('Precision')

    # Gràfic per Recall
    plt.subplot(233)
    plt.plot(mides, recalls, marker='o', linestyle='-', color='r')
    plt.title('Recall per mida de diccionari')
    plt.xlabel('Número de paraules')
    plt.ylabel('Recall')

    # Gràfic per F1-Score
    plt.subplot(234)
    plt.plot(mides, f1_scores, marker='o', linestyle='-', color='c')
    plt.title('F1-Score per mida de diccionari')
    plt.xlabel('Número de paraules')
    plt.ylabel('F1-Score')

    # Gràfic per Specificity
    plt.subplot(235)
    plt.plot(mides, specificities, marker='o', linestyle='-', color='m')
    plt.title('Specificity per mida de diccionari')
    plt.xlabel('Número de paraules')
    plt.ylabel('Specificity')

    # Mostrar totes les gràfiques
    plt.tight_layout()
    plt.show()
"""
    
diferentsMidesDiccionari(df_sense_nans)

AVALUACIÓ DE L'APARTAT 2 DE LA PART B: DIFERENTS MIDES DE DICCIONARI
 
AVALUACIÓ AMB K-FOLD CROSS-VALIDATION:
  Mitjana accuracies: 0.6761359858848799
  Mitjana precisions: 0.7521033417038949
  Mitjana recalls: 0.5248453249460405
  Mitjana F1-Score: 0.6182510614493648
  Mitjana Specificities: 0.8272317936682978
 
  Mida del diccionari:  641008
    Accuracy:  0.6761359858848799
    Precision:  0.7521033417038949
    Recall:  0.5248453249460405
    F1-Score:  0.6182510614493648
    Specificity:  0.8272317936682978
 
AVALUACIÓ AMB K-FOLD CROSS-VALIDATION:
  Mitjana accuracies: 0.6581833175646304
  Mitjana precisions: 0.7319275981065136
  Mitjana recalls: 0.49929634544271184
  Mitjana F1-Score: 0.5931818266840986
  Mitjana Specificities: 0.8168656302402004
 
  Mida del diccionari:  320504
    Accuracy:  0.6581833175646304
    Precision:  0.7319275981065136
    Recall:  0.49929634544271184
    F1-Score:  0.5931818266840986
    Specificity:  0.8168656302402004
 
AVALUACIÓ AMB K-FOLD CROSS-VA

In [9]:
#3. Mida de diccionari fixe, diferents conjunts de train

print("AVALUACIÓ DE L'APARTAT 3 DE LA PART B: MIDA DE DICCIONARI FIXE, DIFERENTS CONJUNTS DE TRAIN")
print(" ")

def dicFixeDiferentsTrain(df, mida = 10000, particions = [0.2, 0.4, 0.6, 0.8, 0.95], laplace = False):

    #llistes per fer gràfiques
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    specificities = []
        
    for particio in particions:
        X_train, X_test, y_train, y_test = train_test_split(df['tweetText'], df['sentimentLabel'], train_size=particio, stratify=df['sentimentLabel'], random_state=None)

        diccionari = fer_diccionari(X_train, y_train) 

        diccionari_reduit = dict(random.sample(list(diccionari.items()), mida))
            
        prob_paraules, prob_classes = calculProbs(y_train, diccionari_reduit, laplace)
        prediccions = [] 

        for twit in X_test:
            prediccions.append(fer_prediccio(twit, prob_classes, prob_paraules))

        accuracy, precision, recall, f1, specificity = metriques(prediccions, y_test)

        accuracies.append(accuracy)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        specificities.append(specificity)
            
            
        print("  Partició: ",particio, " train, ", 1- particio, " test")
        print("    Accuracy: ", accuracy)
        print("    Precision: ", precision)
        print("    Recall: ", recall)
        print("    F1-Score: ", f1)
        print("    Specificity: ", specificity)
        print(" ")

        #corbes(y_test, prediccions)

    
    """
    # Crear les gràfiques per cada mètrica
    plt.figure(figsize=(12, 8))
    
    # Gràfic per Accuracy
    plt.subplot(231)
    plt.plot(particions, accuracies, marker='o', linestyle='-', color='b')
    plt.title('Accuracy per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('Accuracy')

    # Gràfic per Precision
    plt.subplot(232)
    plt.plot(particions, precisions, marker='o', linestyle='-', color='g')
    plt.title('Precision per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('Precision')

    # Gràfic per Recall
    plt.subplot(233)
    plt.plot(particions, recalls, marker='o', linestyle='-', color='r')
    plt.title('Recall per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('Recall')

    # Gràfic per F1-Score
    plt.subplot(234)
    plt.plot(particions, f1_scores, marker='o', linestyle='-', color='c')
    plt.title('F1-Score per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('F1-Score')

    # Gràfic per Specificity
    plt.subplot(235)
    plt.plot(particions, specificities, marker='o', linestyle='-', color='m')
    plt.title('Specificity per Partició')
    plt.xlabel('Proporció train')
    plt.ylabel('Specificity')

    # Mostrar totes les gràfiques
    plt.tight_layout()
    plt.show()"""

 #Creem el diccionari a partir de totes les paraules per no tenir conflictes  de grandàries
#mides = [len(diccionari)//5, len(diccionari)//4, len(diccionari)//3, len(diccionari)//2, len(diccionari)]
mida = 200000
dicFixeDiferentsTrain(df_sense_nans, mida)


AVALUACIÓ DE L'APARTAT 3 DE LA PART B: MIDA DE DICCIONARI FIXE, DIFERENTS CONJUNTS DE TRAIN
 
  Partició:  0.2  train,  0.8  test
    Accuracy:  0.6633882680849976
    Precision:  0.7444495288658041
    Recall:  0.4969239208135498
    F1-Score:  0.596008846213602
    Specificity:  0.8296383252277936
 
  Partició:  0.4  train,  0.6  test
    Accuracy:  0.6735132670195446
    Precision:  0.7044288387629032
    Recall:  0.5971743051972144
    F1-Score:  0.6463825848933367
    Specificity:  0.7497540386638443
 
  Partició:  0.6  train,  0.4  test
    Accuracy:  0.6454503030148055
    Precision:  0.6953280275328028
    Recall:  0.5169580430763719
    F1-Score:  0.5930207813554844
    Specificity:  0.7737767442454753
 
  Partició:  0.8  train,  0.19999999999999996  test
    Accuracy:  0.6191186999769862
    Precision:  0.6718452347925798
    Recall:  0.4647437742680407
    F1-Score:  0.5494262000642807
    Specificity:  0.7732944055095222
 
  Partició:  0.95  train,  0.050000000000000044  te

In [10]:
#Part A

#1.Ampliar particions amb Laplace Smoothing

print("AVALUACIÓ DE L'APARTAT 1 DE LA PART A: AMPLIAR PARTICIONS AMB LAPLACE SMOOTHING")
print()
ampliarParticions(df_sense_nans, laplace = True)

#2.Diferents mides de diccionari

print("AVALUACIÓ DE L'APARTAT 2 DE LA PART A: DIFERENTS MIDES DE DICCIONARI AMB LAPLACE SMOOTHING")
print(" ")
diferentsMidesDiccionari(df_sense_nans, laplace = True)

#3. Mida de diccionari fixe, diferents conjunts de train

mida = 200000
print("AVALUACIÓ DE L'APARTAT 3 DE LA PART A: MIDA DE DICCIOANRI FIXE, DIFERENTS CONJUNTS DE TRAIN AMB LAPLACE SMOOTHING")
print(" ")
dicFixeDiferentsTrain(df_sense_nans, mida, laplace=True)




AVALUACIÓ DE L'APARTAT 1 DE LA PART A: AMPLIAR PARTICIONS AMB LAPLACE SMOOTHING

  Partició:  0.2  train,  0.8  test
    Accuracy:  0.7409383230623674
    Precision:  0.8581054884937861
    Recall:  0.5769451583137297
    F1-Score:  0.6899823473273318
    Specificity:  0.904720378844142
 
  Partició:  0.4  train,  0.6  test
    Accuracy:  0.7460940496586289
    Precision:  0.8590838050050125
    Recall:  0.5883722616219812
    F1-Score:  0.6984130198182693
    Specificity:  0.903612969722266
 
  Partició:  0.6  train,  0.4  test
    Accuracy:  0.7504027411972282
    Precision:  0.8592405528261169
    Recall:  0.5985338425224049
    F1-Score:  0.7055749523041075
    Specificity:  0.9020756537127306
 
  Partició:  0.8  train,  0.19999999999999996  test
    Accuracy:  0.7498082184775104
    Precision:  0.8586038647786896
    Recall:  0.5977278397205857
    F1-Score:  0.7048001930938768
    Specificity:  0.9016923381609797
 
  Partició:  0.95  train,  0.050000000000000044  test
    Accurac